# PVision AI: Solar Power Generation Dataset Inspection
**Project:** A Hybrid CNN-ANN Framework for Photovoltaic Fault Detection and Power Generation Prediction  
**Phase 1:** Dataset Inspection and Exploratory Analysis (CSVs: Generation & Weather Sensors)  

This notebook inspects the solar telemetry datasets for Plant 1 and Plant 2: column definitions, missing values, duplicate rows, descriptive statistics, and weather-to-power correlation.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utilities.config import config
from src.data.inspect_solar_generation import SolarGenerationInspector

print(f"Project Root: {config.PROJECT_ROOT}")
print(f"Solar Generation Data Path: {config.SOLAR_GEN_DATA_DIR}")

## 1. Execute Solar Generation Inspection Pipeline

In [ ]:
inspector = SolarGenerationInspector()
summary = inspector.inspect()

## 2. CSV Overview: Rows, Columns, Missing Values, and Duplicates

In [ ]:
overview_rows = []
for fname, rep in summary.get("file_reports", {}).items():
    overview_rows.append({
        "File Name": fname,
        "Rows": f"{rep['rows']:,}",
        "Columns": rep["columns"],
        "Missing Values": rep["total_missing_values"],
        "Exact Duplicates": rep["exact_duplicate_rows"],
        "Composite Key Dups": rep["composite_key_duplicates"]
    })

df_overview = pd.DataFrame(overview_rows)
display(df_overview)

## 3. Descriptive Statistics for Numerical Features

In [ ]:
for fname, rep in summary.get("file_reports", {}).items():
    print(f"\n=== {fname} Descriptive Statistics ===")
    stats_df = pd.DataFrame(rep["descriptive_statistics"]).T
    display(stats_df[["count", "mean", "std", "min", "median", "max", "skewness"]])

## 4. Cross-Dataset Correlation: Weather Parameters vs Power Generation

In [ ]:
for plant, info in summary.get("merged_plant_analysis", {}).items():
    print(f"=== {plant} Weather-to-Power Correlation Matrix ===")
    corr_df = pd.DataFrame(info.get("cross_correlation_weather_power", {}))
    display(corr_df)
    
    plt.figure(figsize=(7, 5))
    sns.heatmap(corr_df, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".3f")
    plt.title(f"{plant} Weather-to-Power Correlation", fontweight="bold")
    plt.tight_layout()
    plt.show()

## 5. Key Preprocessing Insights for ANN Power Forecasting
1. **Datetime Alignment:** Plant 1 Generation uses `%d-%m-%Y %H:%M` while Weather uses `%Y-%m-%d %H:%M:%S`. Parsing must account for this discrepancy.
2. **Physical Correlations:** Solar irradiance (`IRRADIATION`) and `MODULE_TEMPERATURE` exhibit massive positive correlations with `AC_POWER` and `DC_POWER` (> 0.70-0.90), making them prime predictors for the ANN model.
3. **Inverter Aggregation vs Individual Tracking:** Generation data features 22 distinct inverters (`SOURCE_KEY`) monitored per plant, enabling inverter-level efficiency profiling.